# AI/ML Internship — Week 2: Data Analysis with NumPy & Pandas
## Comprehensive Data Analysis — Titanic Survival Dataset

**Intern:** Abdul Rafay  
**Week:** 2 of 8  
**Instructor:** Zain Ul Abideen  
**Dataset:** Titanic — Machine Learning from Disaster (Kaggle)

---

## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# For missing data visualization
try:
    import missingno as msno
except ImportError:
    !pip install missingno -q
    import missingno as msno

# For feature scaling
from sklearn.preprocessing import StandardScaler

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("All libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---
# PART A — NumPy Deep Dive & Data Loading (Steps 1–4)
---

## Step 1: NumPy Warm-Up — Broadcasting & Masking

Complete NumPy exercises covering broadcasting, boolean masking, fancy indexing, and dot products before touching the dataset.

In [ ]:
np.random.seed(42)

# (a) Create a 6×4 matrix of random integers (0–100): 6 students, 4 exam scores
scores = np.random.randint(0, 101, size=(6, 4))
print("=" * 60)
print("(a) Original 6×4 Score Matrix (6 students, 4 exams)")
print("=" * 60)
print(scores)
print(f"Shape: {scores.shape}")

In [ ]:
# (b) Normalize each column to [0,1] using broadcasting ONLY (no loops)
mins = scores.min(axis=0)   # shape (4,) — min per column
maxs = scores.max(axis=0)   # shape (4,) — max per column
normalized = (scores - mins) / (maxs - mins)  # broadcasting: (6,4) - (4,) / (4,)

print("=" * 60)
print("(b) Normalized Matrix [0, 1] — via Broadcasting")
print("=" * 60)
print(np.round(normalized, 3))
print(f"\nColumn mins (should be 0): {normalized.min(axis=0)}")
print(f"Column maxs (should be 1): {normalized.max(axis=0)}")

In [ ]:
# (c) Boolean mask: students averaging >= 60 across all subjects
student_avgs = scores.mean(axis=1)  # mean per row
mask_pass = student_avgs >= 60
passing_students = scores[mask_pass]

print("=" * 60)
print("(c) Boolean Masking — Students Averaging >= 60")
print("=" * 60)
print(f"Student averages: {student_avgs}")
print(f"Pass mask: {mask_pass}")
print(f"Number of passing students: {mask_pass.sum()} out of {len(scores)}")
print(f"\nPassing students' scores:\n{passing_students}")

In [ ]:
# (d) Replace scores below 40 with the column mean using np.where
col_means = scores.mean(axis=0)  # mean per column
scores_cleaned = np.where(scores < 40, col_means, scores)  # broadcasting

print("=" * 60)
print("(d) np.where — Replace Scores Below 40 with Column Mean")
print("=" * 60)
print(f"Column means: {np.round(col_means, 1)}")
print(f"\nOriginal scores:\n{scores}")
print(f"\nCleaned scores (below 40 replaced):\n{np.round(scores_cleaned, 1)}")

In [ ]:
# (e) Dot product: weighted scoring system
np.random.seed(7)
weights = np.random.rand(4)
weights = weights / weights.sum()  # normalize to sum to 1

weighted_scores = normalized @ weights  # matrix multiply: (6,4) @ (4,) = (6,)

print("=" * 60)
print("(e) Dot Product — Weighted Scoring System")
print("=" * 60)
print(f"Weight vector: {np.round(weights, 3)}")
print(f"Weighted scores per student: {np.round(weighted_scores, 3)}")
print(f"Best student index: {np.argmax(weighted_scores)} (score: {weighted_scores.max():.3f})")
print(f"Worst student index: {np.argmin(weighted_scores)} (score: {weighted_scores.min():.3f})")

## Step 2: Load Dataset & Full Initial Inspection

Loading the Titanic `train.csv` dataset and performing comprehensive initial inspection.

In [ ]:
# Load the Titanic dataset
# Try multiple sources for reliability
try:
    df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
    print("Loaded from GitHub (datasciencedojo)")
except:
    try:
        df = pd.read_csv('https://raw.githubusercontent.com/pandas-dev/pandas/master/doc/data/titanic.csv')
        print("Loaded from pandas GitHub")
    except:
        # Kaggle alternative via seaborn (different column names)
        print("Loading from seaborn built-in dataset...")
        df_sns = sns.load_dataset('titanic')
        # Map to Kaggle column names
        df = df_sns.rename(columns={
            'survived': 'Survived', 'pclass': 'Pclass', 'sex': 'Sex',
            'age': 'Age', 'sibsp': 'SibSp', 'parch': 'Parch',
            'fare': 'Fare', 'embarked': 'Embarked', 'class': 'Class',
            'who': 'Who', 'deck': 'Deck', 'embark_town': 'Embark_town',
            'alive': 'Alive', 'alone': 'Alone'
        })
        # Add missing columns
        df['PassengerId'] = range(1, len(df)+1)
        df['Name'] = 'Unknown'
        df['Ticket'] = 'Unknown'
        df['Cabin'] = df['Deck'].astype(str).replace('nan', np.nan)
        df = df[['PassengerId','Survived','Pclass','Name','Sex','Age',
                 'SibSp','Parch','Ticket','Fare','Cabin','Embarked']]

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape} ({df.shape[0]} rows, {df.shape[1]} columns)")

In [ ]:
# Display: head(10)
print("=" * 70)
print("FIRST 10 ROWS — df.head(10)")
print("=" * 70)
df.head(10)

In [ ]:
# Display: tail(5)
print("=" * 70)
print("LAST 5 ROWS — df.tail(5)")
print("=" * 70)
df.tail(5)

In [ ]:
# Display: sample(8)
print("=" * 70)
print("RANDOM SAMPLE OF 8 ROWS — df.sample(8, random_state=42)")
print("=" * 70)
df.sample(8, random_state=42)

In [ ]:
# Shape, info, dtypes, columns
print("=" * 70)
print("DATASET SHAPE")
print("=" * 70)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)
print(df.dtypes)

print("\n" + "=" * 70)
print("COLUMN LIST")
print("=" * 70)
print(df.columns.tolist())

print("\n" + "=" * 70)
print("DATASET INFO")
print("=" * 70)
df.info()

In [ ]:
# Categorical vs Numerical columns
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()
cols_with_missing = df.columns[df.isnull().any()].tolist()
total_missing = df.isnull().sum().sum()

print("=" * 70)
print("COLUMN ANALYSIS SUMMARY")
print("=" * 70)
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numerical columns ({len(num_cols)}): {num_cols}")
print(f"Columns with ANY missing values ({len(cols_with_missing)}): {cols_with_missing}")
print(f"Total missing cells in entire dataset: {total_missing}")

### Column Descriptions

| Column | Description | Type |
|--------|-------------|------|
| **PassengerId** | Unique identifier for each passenger | Numerical (ID) |
| **Survived** | Survival indicator (0 = No, 1 = Yes) | Binary target |
| **Pclass** | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) | Ordinal categorical |
| **Name** | Full name of the passenger | Text |
| **Sex** | Gender (male/female) | Binary categorical |
| **Age** | Age in years | Continuous numerical |
| **SibSp** | Number of siblings/spouses aboard | Discrete numerical |
| **Parch** | Number of parents/children aboard | Discrete numerical |
| **Ticket** | Ticket number | Text (ID-like) |
| **Fare** | Passenger fare | Continuous numerical |
| **Cabin** | Cabin number | Text (sparse) |
| **Embarked** | Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) | Categorical |

## Step 3: Missing Value Deep Analysis

Analyzing missing values with counts, percentages, and visual patterns.

In [ ]:
# Missing values count and percentage
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print("=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)
print(missing_df[missing_df['Missing Count'] > 0])
print(f"\nTotal missing values: {missing_count.sum()}")

In [ ]:
# Visualize missing data with missingno
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
plt.sca(axes[0])
msno.bar(df, ax=axes[0], fontsize=10, color=(0.27, 0.52, 0.53))
axes[0].set_title('Missing Value Bar Chart', fontsize=13, fontweight='bold')

# Matrix
plt.sca(axes[1])
msno.matrix(df, ax=axes[1], fontsize=10, color=(0.27, 0.52, 0.53))
axes[1].set_title('Missing Value Matrix (white = missing)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### Missing Data Analysis — Three Columns with Most Missing Data

**1. Cabin (687 missing — 77.1%)**
- **Why missing:** Cabin assignments were primarily recorded for upper-class passengers. Third-class passengers often did not have specific cabin assignments, or records were lost during the disaster. Additionally, survival records were prioritized over logistical data.
- **Strategy:** Since 77% of values are missing, imputation is impractical. **Create a binary indicator `has_cabin`** (1 = cabin known, 0 = missing) — this captures the signal (cabin recorded ≈ higher class) without inventing data. Then **drop the original Cabin column**.

**2. Age (177 missing — 19.9%)**
- **Why missing:** Age was self-reported during boarding and not always collected, especially for third-class passengers who boarded in larger groups. Some passengers' manifests were incomplete or lost.
- **Strategy:** **Fill with median age grouped by Pclass and Sex.** Group-level imputation is superior to global mean because age distributions differ significantly across passenger classes and genders — e.g., first-class passengers tended to be older.

**3. Embarked (2 missing — 0.2%)**
- **Why missing:** Minor record-keeping errors. With only 2 missing values, this is negligible.
- **Strategy:** **Fill with mode** (most common port = 'S' for Southampton, where ~72% of passengers boarded).

### Drop vs. Fill Decision
- **Cabin should be dropped** (after creating `has_cabin`) because 77% missing is too high for any imputation method to be reliable — filling would introduce more noise than signal.
- **Age should be filled** (not dropped) because losing 20% of the data would significantly reduce our dataset and bias results toward passengers with recorded ages.
- **Risk of dropping Age:** We'd lose 177 passengers, potentially biasing the analysis toward demographics that had better record-keeping (higher classes). Risk of filling: introduces some inaccuracy, but group-level median minimizes this.

## Step 4: Data Type Audit & Correction

Verifying and fixing data types to match their intended use.

In [ ]:
# Store original dtypes for comparison
original_dtypes = df.dtypes.copy()

print("=" * 70)
print("CURRENT DATA TYPES (Before Correction)")
print("=" * 70)
print(original_dtypes)

In [ ]:
# Fix data types

# 1. Convert 'Survived' to category — it's a binary label, not a number to compute with
df['Survived'] = df['Survived'].astype('category')

# 2. Convert 'Pclass' to category — it's an ordinal class label (1st, 2nd, 3rd), not a number
df['Pclass'] = df['Pclass'].astype('category')

# 3. Convert 'Sex' to category
df['Sex'] = df['Sex'].astype('category')

# 4. Convert 'Embarked' to category
df['Embarked'] = df['Embarked'].astype('category')

# 5. Verify numerical columns are correct type
# Age and Fare should be float — they already are
# SibSp and Parch are integers — correct

print("Data types corrected!")
print("\n" + "=" * 70)
print("UPDATED DATA TYPES (After Correction)")
print("=" * 70)
df.info()

In [ ]:
# Before/After comparison table
new_dtypes = df.dtypes
comparison = pd.DataFrame({
    'Column': original_dtypes.index,
    'Old Dtype': original_dtypes.values.astype(str),
    'New Dtype': new_dtypes.values.astype(str)
})
comparison['Changed'] = comparison['Old Dtype'] != comparison['New Dtype']

print("=" * 70)
print("DATA TYPE COMPARISON: Before vs After")
print("=" * 70)
print(comparison.to_string(index=False))
print(f"\nColumns changed: {comparison['Changed'].sum()}")

### Why Convert Survived and Pclass to Category?

- **Survived** is a binary label (0/1), not a quantity. Computing `mean(Survived)` gives survival rate, which is useful, but the column semantically represents a class membership. Category dtype prevents accidental arithmetic (like `Survived * 2`) and saves memory.
- **Pclass** is ordinal (1st > 2nd > 3rd class), not a continuous number. Computing `mean(Pclass)` is meaningless. Category dtype signals to both humans and ML pipelines that this is a grouping variable, not a regression feature.

> **Note:** We'll convert Survived back to int when needed for numerical computations (like correlation). The category dtype is about semantic correctness during analysis.

---
# PART B — Data Cleaning & Feature Engineering (Steps 5–9)
---

## Step 5: Handle Missing Values — Professional Strategy

In [ ]:
# Convert Survived and Pclass back to numeric for computations
df['Survived'] = df['Survived'].astype(int)
df['Pclass'] = df['Pclass'].astype(int)

# --- AGE: Fill with median grouped by Pclass and Sex ---
print("=" * 70)
print("AGE IMPUTATION — Group-Level Median (by Pclass × Sex)")
print("=" * 70)

# Show group medians first
age_medians = df.groupby(['Pclass', 'Sex'])['Age'].median()
print("Group medians used for imputation:")
print(age_medians)
print(f"\nGlobal median (NOT used): {df['Age'].median()}")

# Fill missing ages with group median
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

print(f"\nAge missing after fill: {df['Age'].isnull().sum()}")

In [ ]:
# --- EMBARKED: Fill with mode ---
print("=" * 70)
print("EMBARKED IMPUTATION — Mode")
print("=" * 70)
print(f"Mode of Embarked: '{df['Embarked'].mode()[0]}'")
print(f"Embarked value counts:\n{df['Embarked'].value_counts()}")

df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
print(f"\nEmbarked missing after fill: {df['Embarked'].isnull().sum()}")

In [ ]:
# --- CABIN: Create binary feature 'has_cabin', then drop Cabin ---
print("=" * 70)
print("CABIN — Binary Indicator + Drop")
print("=" * 70)

df['has_cabin'] = df['Cabin'].notna().astype(int)
print(f"has_cabin distribution:\n{df['has_cabin'].value_counts()}")
print(f"\nSurvival rate WITH cabin: {df[df['has_cabin']==1]['Survived'].mean():.3f}")
print(f"Survival rate WITHOUT cabin: {df[df['has_cabin']==0]['Survived'].mean():.3f}")

# Store original Cabin for later feature engineering (deck extraction)
cabin_original = df['Cabin'].copy()
df = df.drop('Cabin', axis=1)
print(f"\nCabin column dropped. New shape: {df.shape}")

In [ ]:
# --- VERIFICATION: No missing values remain ---
print("=" * 70)
print("VERIFICATION — All Missing Values Handled")
print("=" * 70)
print(f"Total missing values: {df.isnull().sum().sum()}")
assert df.isnull().sum().sum() == 0, "ERROR: Missing values still exist!"
print("✓ CONFIRMED: Zero missing values in the dataset!")

# Summary table
print("\n" + "=" * 70)
print("CLEANING STRATEGY SUMMARY")
print("=" * 70)
summary = pd.DataFrame({
    'Column': ['Age', 'Embarked', 'Cabin'],
    'Missing Count': [177, 2, 687],
    'Missing %': ['19.9%', '0.2%', '77.1%'],
    'Strategy': [
        'Median grouped by Pclass × Sex',
        'Mode (S = Southampton)',
        'Binary indicator has_cabin → drop original'
    ]
})
print(summary.to_string(index=False))

### Why Group-Level Imputation is Superior to Global Mean

Using the **global mean/median** for Age (~28) would assign the same value to a 1st-class woman and a 3rd-class man — ignoring the real demographic differences. Group medians capture that:
- 1st-class passengers were generally older (median ~38–40)
- 3rd-class passengers were younger (median ~22–25)  
- Women in 1st class tended to be slightly younger than men in 1st class

This preserves the relationship between Age and other features, which is critical for any downstream ML model.

## Step 6: Outlier Detection & Treatment

In [ ]:
# Outlier analysis for Fare and Age
print("=" * 70)
print("OUTLIER DETECTION — IQR Method")
print("=" * 70)

for col in ['Fare', 'Age']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    print(f"\n--- {col} ---")
    print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
    print(f"Lower fence: {lower:.2f}")
    print(f"Upper fence: {upper:.2f}")
    print(f"Outliers found: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
    if len(outliers) > 0:
        print(f"Outlier range: [{outliers[col].min():.2f}, {outliers[col].max():.2f}]")

In [ ]:
# Box plots: Before and After treatment
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fare — BEFORE
axes[0, 0].boxplot(df['Fare'], vert=True)
axes[0, 0].set_title('Fare — BEFORE Treatment', fontweight='bold')
axes[0, 0].set_ylabel('Fare ($)')

# Age — BEFORE
axes[0, 1].boxplot(df['Age'].dropna(), vert=True)
axes[0, 1].set_title('Age — BEFORE Treatment', fontweight='bold')
axes[0, 1].set_ylabel('Age (years)')

# Cap Fare outliers at 99th percentile (winsorization)
fare_99 = df['Fare'].quantile(0.99)
print(f"Fare 99th percentile: ${fare_99:.2f}")
print(f"Fares > $300 before capping: {(df['Fare'] > 300).sum()}")

df['Fare'] = df['Fare'].clip(upper=fare_99)
print(f"Fares > $300 after capping: {(df['Fare'] > 300).sum()}")

# Fare — AFTER
axes[1, 0].boxplot(df['Fare'], vert=True)
axes[1, 0].set_title('Fare — AFTER Capping at 99th Percentile', fontweight='bold')
axes[1, 0].set_ylabel('Fare ($)')

# Age — AFTER (no treatment needed)
axes[1, 1].boxplot(df['Age'].dropna(), vert=True)
axes[1, 1].set_title('Age — No Treatment Needed', fontweight='bold')
axes[1, 1].set_ylabel('Age (years)')

plt.suptitle('Outlier Treatment: Before & After', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Why Winsorization (Capping) is Better Than Dropping Outliers in ML

**Capping (winsorization)** limits extreme values to a threshold (e.g., 99th percentile) without removing rows. This is preferred over dropping because:

1. **Data preservation:** Dropping outliers reduces the training set. With 891 rows, losing even 10–20 passengers hurts model performance.
2. **Real signal retention:** A passenger who paid $512 genuinely was first-class — that information matters. Capping to ~$263 preserves the "high fare" signal without the extreme value distorting scaled features.
3. **No bias introduction:** Dropping outliers selectively removes wealthy/first-class passengers, biasing the dataset toward lower classes.
4. **Age outliers:** The Titanic had passengers from infants to 80-year-olds — these are legitimate values, not measurement errors. No treatment is needed.

## Step 7: Feature Engineering — 7 New Columns

In [ ]:
# (a) family_size = SibSp + Parch + 1
df['family_size'] = df['SibSp'] + df['Parch'] + 1

# (b) is_alone = 1 if family_size == 1
df['is_alone'] = (df['family_size'] == 1).astype(int)

# (c) fare_per_person = Fare / family_size
df['fare_per_person'] = df['Fare'] / df['family_size']

# (d) title: extract from Name
df['title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
print("Title value counts (raw):")
print(df['title'].value_counts())

# Group rare titles
rare_titles = df['title'].value_counts()
rare_map = rare_titles[rare_titles < 10].index.tolist()
df['title'] = df['title'].replace(rare_map, 'Rare')

print("\nTitle value counts (after grouping rare):")
print(df['title'].value_counts())

In [ ]:
# (e) age_group: categorize Age into bins
df['age_group'] = pd.cut(df['Age'], bins=[0, 12, 17, 60, 100],
                         labels=['Child', 'Teen', 'Adult', 'Senior'])

# (f) deck: extract first letter of original Cabin
df['deck'] = cabin_original.str[0]  # NaN stays NaN
df['deck'] = df['deck'].fillna('Unknown')
print("Deck distribution:")
print(df['deck'].value_counts())

# (g) fare_bin: quartile-based binning
df['fare_bin'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Medium', 'High', 'VHigh'])

In [ ]:
# Print value_counts for each new categorical feature
print("=" * 70)
print("VALUE COUNTS FOR ALL NEW FEATURES")
print("=" * 70)

for feat in ['family_size', 'is_alone', 'title', 'age_group', 'deck', 'fare_bin']:
    print(f"\n--- {feat} ---")
    print(df[feat].value_counts())

print("\n" + "=" * 70)
print("FIRST 10 ROWS — All 7 New Features")
print("=" * 70)
new_cols = ['family_size', 'is_alone', 'fare_per_person', 'title', 'age_group', 'deck', 'fare_bin']
df[new_cols].head(10)

## Step 8: Encoding Categorical Variables

In [ ]:
print("=" * 70)
print("CATEGORICAL ENCODING")
print("=" * 70)

# (a) Sex: Label encode — Male=0, Female=1
df['sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})
print("(a) Sex → sex_encoded: male=0, female=1")
print(df[['Sex', 'sex_encoded']].head())

# (b) Embarked: One-Hot Encode
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', dtype=int)
df = pd.concat([df, embarked_dummies], axis=1)
print("\n(b) Embarked → One-Hot Encoded:")
print(df[['Embarked', 'Embarked_C', 'Embarked_Q', 'Embarked_S']].head())

# (c) Title: One-Hot Encode — drop most common to avoid dummy variable trap
title_dummies = pd.get_dummies(df['title'], prefix='title', drop_first=True, dtype=int)
df = pd.concat([df, title_dummies], axis=1)
print(f"\n(c) Title one-hot columns: {title_dummies.columns.tolist()}")

# (d) age_group and fare_bin: Ordinal encode
age_map = {'Child': 0, 'Teen': 1, 'Adult': 2, 'Senior': 3}
fare_map = {'Low': 0, 'Medium': 1, 'High': 2, 'VHigh': 3}

df['age_group_encoded'] = df['age_group'].map(age_map)
df['fare_bin_encoded'] = df['fare_bin'].map(fare_map)

print(f"\n(d) age_group mapping: {age_map}")
print(f"    fare_bin mapping: {fare_map}")

print(f"\nDataset shape after encoding: {df.shape}")
print(f"Columns added by encoding: {df.shape[1] - 12}")  # original had 12 cols

### Why Binary Encoding Works for Sex

`Sex` has exactly **two** categories (male/female), so a single column with 0/1 captures all the information — one-hot encoding would create two perfectly correlated columns (one is always 1 minus the other), which is redundant and introduces the **dummy variable trap** in linear models.

## Step 9: Feature Scaling & Final ML-Ready Dataset

In [ ]:
# Identify continuous features to scale
continuous_features = ['Age', 'Fare', 'fare_per_person', 'family_size']

print("=" * 70)
print("FEATURE SCALING — StandardScaler")
print("=" * 70)

# Before scaling stats
print("\nBEFORE Scaling:")
for col in continuous_features:
    print(f"  {col}: mean={df[col].mean():.3f}, std={df[col].std():.3f}")

# Apply StandardScaler
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

# After scaling stats
print("\nAFTER Scaling:")
for col in continuous_features:
    print(f"  {col}: mean={df[col].mean():.6f}, std={df[col].std():.6f}")
print("\n✓ Means ≈ 0 and Stds ≈ 1 confirmed!")

In [ ]:
# Create final ML-ready dataset
drop_cols = ['Name', 'Ticket', 'PassengerId', 'Sex', 'Embarked', 
             'title', 'age_group', 'fare_bin', 'deck']
final_df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print("=" * 70)
print("FINAL ML-READY DATASET")
print("=" * 70)
print(f"Shape: {final_df.shape}")
print(f"\nColumns: {final_df.columns.tolist()}")
final_df.info()

In [ ]:
# Display first rows
print("\nFirst 5 rows of final dataset:")
final_df.head()

In [ ]:
# Save cleaned dataset
final_df.to_csv('titanic_cleaned.csv', index=False)
print("✓ Saved: titanic_cleaned.csv")

---
# PART C — Advanced GroupBy & Statistical Analysis (Steps 10–13)
---

## Step 10: Survival Analysis — Multi-Factor GroupBy

In [ ]:
# Reload original for cleaner analysis (keep engineered features)
# Re-read from CSV we just cleaned, or reload from source
try:
    df_analysis = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
except:
    # Fallback: reconstruct from our working copy
    df_analysis = pd.read_csv('titanic_cleaned.csv')
    # We need original columns, so let's just use what we have
    print("Using cleaned dataset for analysis")

# Re-apply cleaning
df_analysis['Age'] = df_analysis.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
df_analysis['Embarked'] = df_analysis['Embarked'].fillna('S')
df_analysis['has_cabin'] = df_analysis['Cabin'].notna().astype(int)
df_analysis['family_size'] = df_analysis['SibSp'] + df_analysis['Parch'] + 1
df_analysis['is_alone'] = (df_analysis['family_size'] == 1).astype(int)
df_analysis['title'] = df_analysis['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare_titles = df_analysis['title'].value_counts()
df_analysis['title'] = df_analysis['title'].replace(
    rare_titles[rare_titles < 10].index.tolist(), 'Rare'
)
df_analysis['age_group'] = pd.cut(df_analysis['Age'], bins=[0, 12, 17, 60, 100],
                                   labels=['Child', 'Teen', 'Adult', 'Senior'])

overall_survival = df_analysis['Survived'].mean()
print(f"Overall survival rate: {overall_survival:.3f}")

In [ ]:
# Create 6-panel survival analysis
fig, axes = plt.subplots(3, 2, figsize=(15, 18))
colors = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6', '#1abc9c']

# (a) By Pclass
surv_class = df_analysis.groupby('Pclass')['Survived'].mean()
ax = axes[0, 0]
bars = ax.bar(surv_class.index.astype(str), surv_class.values, color=colors[:3], edgecolor='black')
for bar, val in zip(bars, surv_class.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', fontweight='bold')
ax.axhline(y=overall_survival, color='red', linestyle='--', label=f'Overall: {overall_survival:.3f}')
ax.set_title('(a) Survival Rate by Passenger Class', fontweight='bold')
ax.set_xlabel('Pclass')
ax.set_ylabel('Survival Rate')
ax.legend()

# (b) By Sex
surv_sex = df_analysis.groupby('Sex')['Survived'].mean()
ax = axes[0, 1]
bars = ax.bar(surv_sex.index, surv_sex.values, color=['#3498db', '#e74c3c'], edgecolor='black')
for bar, val in zip(bars, surv_sex.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', fontweight='bold')
ax.axhline(y=overall_survival, color='red', linestyle='--', label=f'Overall: {overall_survival:.3f}')
ax.set_title('(b) Survival Rate by Sex', fontweight='bold')
ax.set_ylabel('Survival Rate')
ax.legend()

# (c) By Pclass + Sex (unstacked)
surv_cls_sex = df_analysis.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack()
ax = axes[1, 0]
surv_cls_sex.plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c'], edgecolor='black')
ax.axhline(y=overall_survival, color='green', linestyle='--', label=f'Overall: {overall_survival:.3f}')
ax.set_title('(c) Survival Rate by Class × Sex', fontweight='bold')
ax.set_xlabel('Pclass')
ax.set_ylabel('Survival Rate')
ax.legend()
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Print the table
print("Survival by Pclass × Sex:")
print(surv_cls_sex.round(3))

# (d) By age_group
surv_age = df_analysis.groupby('age_group', observed=True)['Survived'].mean()
ax = axes[1, 1]
bars = ax.bar(surv_age.index.astype(str), surv_age.values, color=colors[:4], edgecolor='black')
for bar, val in zip(bars, surv_age.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', fontweight='bold')
ax.axhline(y=overall_survival, color='red', linestyle='--', label=f'Overall: {overall_survival:.3f}')
ax.set_title('(d) Survival Rate by Age Group', fontweight='bold')
ax.set_ylabel('Survival Rate')
ax.legend()

# (e) By is_alone
surv_alone = df_analysis.groupby('is_alone')['Survived'].mean()
ax = axes[2, 0]
bars = ax.bar(['With Family', 'Alone'], surv_alone.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
for bar, val in zip(bars, surv_alone.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', fontweight='bold')
ax.axhline(y=overall_survival, color='blue', linestyle='--', label=f'Overall: {overall_survival:.3f}')
ax.set_title('(e) Survival Rate: Alone vs With Family', fontweight='bold')
ax.set_ylabel('Survival Rate')
ax.legend()

# (f) By Embarked
surv_emb = df_analysis.groupby('Embarked')['Survived'].mean()
ax = axes[2, 1]
bars = ax.bar(surv_emb.index, surv_emb.values, color=colors[:3], edgecolor='black')
for bar, val in zip(bars, surv_emb.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
            ha='center', fontweight='bold')
ax.axhline(y=overall_survival, color='red', linestyle='--', label=f'Overall: {overall_survival:.3f}')
ax.set_title('(f) Survival Rate by Embarkation Port', fontweight='bold')
ax.set_ylabel('Survival Rate')
ax.legend()

plt.suptitle('Step 10: Multi-Factor Survival Analysis', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Strongest Predictor of Survival: **Sex**

**Sex is the single strongest predictor of survival.** The survival rate for females was **0.742** compared to just **0.189** for males — a difference of **55.3 percentage points**. This is the largest gap produced by any single factor:

- Pclass: 1st class (0.630) vs 3rd class (0.242) → gap of ~39 points
- is_alone: With family (0.506) vs Alone (0.304) → gap of ~20 points
- Age group: Children (~0.54) vs Adults (~0.36) → gap of ~18 points
- Embarked: Cherbourg (0.554) vs Southampton (0.337) → gap of ~22 points

The "women and children first" protocol during the Titanic disaster is clearly reflected in the data. Sex dominates because it directly determined who was given priority access to lifeboats.

## Step 11: Advanced Aggregation — agg() with Custom Functions

In [ ]:
# Custom aggregation
print("=" * 70)
print("ADVANCED AGGREGATION BY PCLASS")
print("=" * 70)

def pct_above_50(x):
    return (x > 50).mean() * 100

def iqr(x):
    return x.quantile(0.75) - x.quantile(0.25)

agg_result = df_analysis.groupby('Pclass').agg(
    fare_mean=('Fare', 'mean'),
    fare_median=('Fare', 'median'),
    fare_std=('Fare', 'std'),
    fare_min=('Fare', 'min'),
    fare_max=('Fare', 'max'),
    fare_pct_above_50=('Fare', pct_above_50),
    age_mean=('Age', 'mean'),
    age_median=('Age', 'median'),
    age_iqr=('Age', iqr),
    survival_rate=('Survived', 'mean'),
    passenger_count=('Survived', 'count')
).round(3)

print(agg_result)

In [ ]:
# Transform: add group statistics to each row
df_analysis['class_avg_fare'] = df_analysis.groupby('Pclass')['Fare'].transform('mean')
df_analysis['class_survival_rate'] = df_analysis.groupby('Pclass')['Survived'].transform('mean')

print("=" * 70)
print("TRANSFORM — Group Stats Broadcast to Each Row")
print("=" * 70)
print("\nFirst 15 rows with class_avg_fare and class_survival_rate:")
print(df_analysis[['Pclass', 'Fare', 'class_avg_fare', 'Survived', 'class_survival_rate']].head(15).to_string())

## Step 12: Pivot Table Analysis

In [ ]:
# (a) Pivot: Pclass × Sex → Survived mean
print("=" * 70)
print("PIVOT TABLE (a): Survival Rate by Pclass × Sex")
print("=" * 70)

pivot_a = pd.pivot_table(df_analysis, values='Survived', index='Pclass',
                          columns='Sex', aggfunc='mean')
print(pivot_a.round(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.heatmap(pivot_a, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[0],
            linewidths=1, vmin=0, vmax=1)
axes[0].set_title('(a) Survival Rate: Pclass × Sex', fontweight='bold')

In [ ]:
# (b) Pivot: age_group × Pclass → Survived mean and count
print("=" * 70)
print("PIVOT TABLE (b): Survival by Age Group × Pclass")
print("=" * 70)

pivot_b = pd.pivot_table(df_analysis, values='Survived', index='age_group',
                          columns='Pclass', aggfunc=['mean', 'count'])
print(pivot_b.round(3))

In [ ]:
# (c) Pivot: title × Pclass → Fare median
print("=" * 70)
print("PIVOT TABLE (c): Median Fare by Title × Pclass")
print("=" * 70)

pivot_c = pd.pivot_table(df_analysis, values='Fare', index='title',
                          columns='Pclass', aggfunc='median')
print(pivot_c.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(pivot_c, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0],
            linewidths=1)
axes[0].set_title('(c) Median Fare: Title × Pclass', fontweight='bold')

# Also visualize pivot_a more clearly
sns.heatmap(pd.pivot_table(df_analysis, values='Survived', index='Pclass',
                            columns='Sex', aggfunc='mean'),
            annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1],
            linewidths=1, vmin=0, vmax=1)
axes[1].set_title('(a) Survival Rate: Pclass × Sex', fontweight='bold')

plt.tight_layout()
plt.show()

### Pivot Table Interpretation

**(a) Pclass × Sex Survival:**
- **Highest survival:** 1st-class females at **96.8%** — nearly every first-class woman survived.
- **Lowest survival:** 3rd-class males at **13.5%** — the most disadvantaged group.
- The "women and children first" policy combined with class-based access to lifeboats (upper decks = closer to boats) created this dramatic disparity.

**(b) Age Group × Pclass:**
- Children in 1st and 2nd class had near-100% survival. Children in 3rd class had much lower rates, reflecting the structural disadvantage of being on lower decks during the sinking.

**(c) Median Fare × Title:**
- "Mrs" and "Miss" in 1st class paid the highest median fares, reflecting the wealth of traveling families. "Mr" titles consistently paid less, possibly because many male passengers were traveling alone for business.

## Step 13: Correlation Analysis — Full Feature Set

In [ ]:
# Prepare numerical dataset for correlation
# Use the final_df but ensure Survived is included
corr_df = final_df.select_dtypes(include=[np.number])

# Compute correlation matrix
corr_matrix = corr_df.corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, mask=mask, square=True,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Matrix — All Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 features correlated with Survived
print("=" * 70)
print("FEATURE-SURVIVAL CORRELATIONS (Ranked)")
print("=" * 70)

if 'Survived' in corr_matrix.columns:
    surv_corr = corr_matrix['Survived'].drop('Survived').abs().sort_values(ascending=False)
    print("\nTop features by |correlation| with Survived:")
    for i, (feat, val) in enumerate(surv_corr.items(), 1):
        direction = "+" if corr_matrix.loc[feat, 'Survived'] > 0 else "-"
        print(f"  {i}. {feat}: {direction}{val:.3f}")
    
    print(f"\n--- Top 5 for ML ---")
    top5 = surv_corr.head(5).index.tolist()
    print(f"  {top5}")
else:
    print("Survived column not found in correlation matrix")

In [ ]:
# Multicollinearity check
print("\n" + "=" * 70)
print("MULTICOLLINEARITY CHECK (|r| > 0.7)")
print("=" * 70)

high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    for c1, c2, r in high_corr_pairs:
        print(f"  {c1} ↔ {c2}: r = {r:.3f}")
else:
    print("  No pairs with |r| > 0.7 found.")

# Near-zero correlations with Survived
print("\n" + "=" * 70)
print("NEAR-ZERO CORRELATIONS WITH SURVIVED (candidates for removal)")
print("=" * 70)
if 'Survived' in corr_matrix.columns:
    weak = corr_matrix['Survived'].drop('Survived').abs()
    weak_feats = weak[weak < 0.05].sort_values()
    if len(weak_feats) > 0:
        for feat, val in weak_feats.items():
            print(f"  {feat}: {val:.4f}")
    else:
        print("  No features with |r| < 0.05")

### Feature Selection Recommendation for Logistic Regression (Week 5)

Based on correlation analysis, the **top 5 features** for a logistic regression classifier:

1. **sex_encoded** — Strongest predictor; directly determines lifeboat priority
2. **Pclass** — Strong negative correlation; class determined deck location and access
3. **Fare / fare_per_person** — Proxy for wealth and class
4. **has_cabin** — Correlated with class; indicates record quality
5. **family_size / is_alone** — Traveling with family affected survival odds

**Multicollinearity note:** If Fare and Pclass are both included, watch for multicollinearity since higher-class tickets cost more. Consider keeping `fare_per_person` instead of raw `Fare` to reduce redundancy.

---
# PART D — Final Dashboard, NumPy Analysis & Report (Steps 14–16)
---

## Step 14: NumPy Performance & Survival Computation

In [ ]:
# Extract numerical features as NumPy array
num_array = final_df.select_dtypes(include=[np.number]).values
col_names = final_df.select_dtypes(include=[np.number]).columns.tolist()

print("=" * 70)
print("(a) NUMPY DESCRIPTIVE STATISTICS — Every Column")
print("=" * 70)
print(f"Array shape: {num_array.shape}")

stats_table = pd.DataFrame({
    'Column': col_names,
    'Mean': np.mean(num_array, axis=0).round(4),
    'Std': np.std(num_array, axis=0).round(4),
    'Min': np.min(num_array, axis=0).round(4),
    'Max': np.max(num_array, axis=0).round(4),
    'Median': np.median(num_array, axis=0).round(4)
})
print(stats_table.to_string(index=False))

In [ ]:
# (b) Z-score matrix
print("=" * 70)
print("(b) Z-SCORE MATRIX")
print("=" * 70)

means = np.mean(num_array, axis=0)
stds = np.std(num_array, axis=0)
# Avoid division by zero for constant columns
stds_safe = np.where(stds == 0, 1, stds)
z_scores = (num_array - means) / stds_safe

print(f"Z-score matrix shape: {z_scores.shape}")
print(f"Column means after z-scoring (should be ≈0): {np.mean(z_scores, axis=0).round(6)}")
print(f"Column stds after z-scoring (should be ≈1): {np.std(z_scores, axis=0).round(6)}")

In [ ]:
# (c) np.corrcoef vs pandas .corr()
print("=" * 70)
print("(c) CORRELATION COMPARISON: NumPy vs Pandas")
print("=" * 70)

numpy_corr = np.corrcoef(num_array.T)
pandas_corr = final_df.select_dtypes(include=[np.number]).corr().values

# Check if they match
max_diff = np.max(np.abs(numpy_corr - pandas_corr))
print(f"Max absolute difference between NumPy and Pandas correlation: {max_diff:.2e}")
print(f"Matrices are {'identical' if max_diff < 1e-10 else 'different'}!")

In [ ]:
# (d) Survivors vs Non-Survivors comparison
print("=" * 70)
print("(d) SURVIVORS vs NON-SURVIVORS — NumPy Boolean Indexing")
print("=" * 70)

# Find the Survived column index
surv_idx = col_names.index('Survived') if 'Survived' in col_names else None

if surv_idx is not None:
    survived_mask = num_array[:, surv_idx] == 1
    not_survived_mask = num_array[:, surv_idx] == 0
    
    # Find Age and Fare column indices
    age_idx = col_names.index('Age') if 'Age' in col_names else None
    fare_idx = col_names.index('Fare') if 'Fare' in col_names else None
    
    if age_idx is not None and fare_idx is not None:
        surv_age = np.mean(num_array[survived_mask, age_idx])
        not_surv_age = np.mean(num_array[not_survived_mask, age_idx])
        surv_fare = np.mean(num_array[survived_mask, fare_idx])
        not_surv_fare = np.mean(num_array[not_survived_mask, fare_idx])
        
        print(f"{'Metric':<25} {'Survived':>12} {'Not Survived':>14} {'Difference':>12}")
        print("-" * 65)
        print(f"{'Mean Age (scaled)':<25} {surv_age:>12.4f} {not_surv_age:>14.4f} {surv_age - not_surv_age:>12.4f}")
        print(f"{'Mean Fare (scaled)':<25} {surv_fare:>12.4f} {not_surv_fare:>14.4f} {surv_fare - not_surv_fare:>12.4f}")
else:
    print("Note: Survived column position not found, using df_analysis instead")
    surv = df_analysis[df_analysis['Survived'] == 1]
    not_surv = df_analysis[df_analysis['Survived'] == 0]
    print(f"{'Metric':<25} {'Survived':>12} {'Not Survived':>14} {'Difference':>12}")
    print("-" * 65)
    print(f"{'Mean Age':<25} {surv['Age'].mean():>12.2f} {not_surv['Age'].mean():>14.2f} {surv['Age'].mean() - not_surv['Age'].mean():>12.2f}")
    print(f"{'Mean Fare':<25} {surv['Fare'].mean():>12.2f} {not_surv['Fare'].mean():>14.2f} {surv['Fare'].mean() - not_surv['Fare'].mean():>12.2f}")

### Analysis: Fare Difference Between Survivors and Non-Survivors

Survivors paid significantly higher fares on average than non-survivors. This reflects the strong correlation between wealth (fare), passenger class, and survival. Wealthier passengers in upper classes had cabins on higher decks — physically closer to the lifeboats — and received preferential treatment during the evacuation. The fare difference is essentially a proxy for the class-based survival disparity observed throughout this analysis.

## Step 15: Professional 6-Chart Titanic EDA Dashboard

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# ---- Chart 1: Survival rate bar chart by Pclass ----
ax = axes[0, 0]
surv_by_class = df_analysis.groupby('Pclass')['Survived'].mean()
colors_class = ['#2ecc71', '#f39c12', '#e74c3c']
bars = ax.bar([1, 2, 3], surv_by_class.values, color=colors_class, edgecolor='black', width=0.6)
for bar, val in zip(bars, surv_by_class.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.1%}', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Chart 1: Survival Rate by Passenger Class', fontweight='bold', fontsize=12)
ax.set_xlabel('Passenger Class')
ax.set_ylabel('Survival Rate')
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.set_ylim(0, 0.75)

# ---- Chart 2: Age distribution — KDE for survivors vs non-survivors ----
ax = axes[0, 1]
df_analysis[df_analysis['Survived']==1]['Age'].plot.kde(ax=ax, label='Survived', color='#2ecc71', linewidth=2)
df_analysis[df_analysis['Survived']==0]['Age'].plot.kde(ax=ax, label='Not Survived', color='#e74c3c', linewidth=2)
ax.fill_between([], [], alpha=0.3)
ax.set_title('Chart 2: Age Distribution — Survivors vs Non-Survivors', fontweight='bold', fontsize=12)
ax.set_xlabel('Age')
ax.set_ylabel('Density')
ax.legend(fontsize=10)
ax.set_xlim(0, 80)

# ---- Chart 3: Fare distribution box plots by Pclass (log scale) ----
ax = axes[1, 0]
data_by_class = [df_analysis[df_analysis['Pclass']==c]['Fare'].dropna() for c in [1, 2, 3]]
bp = ax.boxplot(data_by_class, labels=['1st Class', '2nd Class', '3rd Class'],
                patch_artist=True)
for patch, color in zip(bp['boxes'], colors_class):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_yscale('log')
ax.set_title('Chart 3: Fare Distribution by Class (Log Scale)', fontweight='bold', fontsize=12)
ax.set_ylabel('Fare (log scale)')
ax.set_xlabel('Passenger Class')

# ---- Chart 4: Heatmap of survival rate by Pclass × Sex ----
ax = axes[1, 1]
pivot_surv = pd.pivot_table(df_analysis, values='Survived', index='Pclass',
                             columns='Sex', aggfunc='mean')
sns.heatmap(pivot_surv, annot=True, fmt='.1%', cmap='RdYlGn', ax=ax,
            linewidths=2, vmin=0, vmax=1, annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Chart 4: Survival Rate — Class × Sex', fontweight='bold', fontsize=12)

# ---- Chart 5: Family size vs survival rate — line plot ----
ax = axes[2, 0]
fam_surv = df_analysis.groupby('family_size')['Survived'].agg(['mean', 'count', 'std'])
fam_surv = fam_surv[fam_surv['count'] >= 5]  # filter small groups
ax.plot(fam_surv.index, fam_surv['mean'], 'o-', color='#3498db', linewidth=2, markersize=8)
# Confidence band
se = fam_surv['std'] / np.sqrt(fam_surv['count'])
ax.fill_between(fam_surv.index, fam_surv['mean'] - 1.96*se, fam_surv['mean'] + 1.96*se,
                alpha=0.2, color='#3498db')
ax.set_title('Chart 5: Family Size vs Survival Rate', fontweight='bold', fontsize=12)
ax.set_xlabel('Family Size')
ax.set_ylabel('Survival Rate')
ax.set_xticks(range(1, int(fam_surv.index.max()) + 1))

# ---- Chart 6: Stacked bar — survival proportion by title ----
ax = axes[2, 1]
title_surv = df_analysis.groupby('title')['Survived'].value_counts(normalize=True).unstack()
title_surv = title_surv.sort_values(by=1, ascending=False) if 1 in title_surv.columns else title_surv
title_surv.plot(kind='bar', stacked=True, ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='black')
ax.set_title('Chart 6: Survival Proportion by Title', fontweight='bold', fontsize=12)
ax.set_xlabel('Title')
ax.set_ylabel('Proportion')
ax.legend(['Not Survived', 'Survived'], loc='upper right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('Titanic EDA Dashboard — Abdul Rafay | Week 2',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('titanic_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✓ Dashboard saved as 'titanic_dashboard.png'")

## Step 16: Written Analysis Report

---

### 1. Executive Summary (100+ words)

This report presents a comprehensive Exploratory Data Analysis (EDA) of the Titanic passenger dataset containing 891 records with 12 features. The analysis covers data cleaning, feature engineering, and multi-factor survival analysis. **Three key findings:** (1) Sex was the strongest single predictor of survival — 74.2% of women survived compared to only 18.9% of men, reflecting the "women and children first" evacuation protocol. (2) Passenger class created dramatic survival disparities: 63% of 1st-class passengers survived vs. only 24.2% of 3rd-class, driven by deck proximity to lifeboats. (3) Family size had a non-linear relationship with survival — passengers in families of 2–4 had the highest survival rates, while solo travelers and very large families fared worse.

### 2. Data Quality Assessment (80+ words)

The dataset had three columns with missing values: Age (19.9%), Cabin (77.1%), and Embarked (0.2%). Age was imputed using group-level median by Pclass and Sex, preserving the demographic structure rather than introducing a single global value. Cabin was converted to a binary indicator (`has_cabin`) because 77% missing makes direct imputation unreliable — but the missingness itself carries signal (cabin records correlate with class). Embarked's 2 missing values were filled with the mode ('S'). Fare outliers above the 99th percentile were capped via winsorization rather than dropped to preserve data points while limiting extreme values.

### 3. Feature Engineering Rationale (80+ words)

Seven features were engineered to improve ML model performance: **family_size** (SibSp + Parch + 1) captures the non-linear effect of family on survival; **is_alone** simplifies this to a binary signal. **fare_per_person** adjusts for family-ticket sharing, providing a more accurate wealth indicator. **title** (extracted via regex from Name) encodes social status and gender in a single variable — "Mrs" and "Miss" had far higher survival than "Mr." **age_group** bins continuous age into meaningful life-stage categories. **deck** captures vertical position on the ship. **fare_bin** quartiles enable ordinal analysis of wealth effects.

### 4. Key Statistical Findings (120+ words)

The survival rate across the dataset was 38.4%. Sex dominated all other predictors: female survival (74.2%) was 4x higher than male (18.9%). Within class-sex combinations, 1st-class females had 96.8% survival while 3rd-class males had only 13.5% — a 7x disparity. Children (<12) had elevated survival rates (57.5%), but only in 1st and 2nd class; 3rd-class children still faced high mortality. Solo travelers had a survival rate of 30.4% compared to 50.6% for those with family, suggesting that having companions improved coordination during evacuation. Passengers boarding at Cherbourg (port C) had the highest survival rate (55.4%), which correlates with a higher proportion of 1st-class passengers at that port. The average fare for survivors was significantly higher than for non-survivors, confirming the wealth-survival relationship.

### 5. Visualization Insights (100+ words)

**Chart 1 (Survival by Class):** Shows the clear class gradient — 1st class had 2.6x the survival of 3rd class. **Chart 2 (Age KDE):** Reveals that very young passengers (0–5) survived at higher rates, and the non-survivor peak is slightly older than the survivor peak. **Chart 3 (Fare Boxplots):** Log-scale boxplots show 1st-class fare range is dramatically wider and higher; even 1st-class outliers are informative, not noise. **Chart 4 (Heatmap):** The most informative single visualization — clearly shows the Sex×Class interaction. **Chart 5 (Family Size):** The inverted-U shape is notable — families of 2–4 optimized survival, while large families (7+) had near-zero rates. **Chart 6 (Title Bars):** "Mrs" and "Miss" titles had the highest survival proportions, while "Mr" had the lowest.

### 6. Feature Selection Recommendation (80+ words)

For a logistic regression model in Week 5, I recommend these 5–7 features: **(1) sex_encoded** — strongest individual predictor, (2) **Pclass** — captures class-based access disparities, (3) **fare_per_person** — better than raw Fare as it adjusts for family ticket sharing, (4) **Age** — captures the "children first" effect, (5) **is_alone** — clean binary signal for family presence, (6) **has_cabin** — proxy for class and record completeness, (7) **Embarked_C** — Cherbourg passengers had notably different demographics. Avoid including both Fare and Pclass simultaneously due to high correlation (multicollinearity risk).

### 7. Reflection (80+ words)

The hardest concept this week was understanding **NumPy broadcasting** — the rules for dimension compatibility (equal or one is 1) seem simple but become subtle with higher-dimensional arrays. My biggest surprise was how dramatically class and sex interacted: I expected class to matter, but 1st-class women having 97% survival while 3rd-class men had 14% was striking. If I could redo this analysis, I would spend more time on **feature interaction terms** (e.g., Pclass × Sex as a combined feature) rather than treating each feature independently, and I would explore the Cabin column's letter (deck) more carefully as a spatial predictor of survival.

---
## End of Week 2 Assignment
**Completed by:** Abdul Rafay  
**Date:** May 2026  
**Tools:** Python, NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn  
---